In [ ]:
# -*- coding: utf-8 -*-
"""
Berramdane Model V11.1 – Final Professional Arabic Educational Tool
Author : Al Moalim Berramdane
License: CC BY 4.0
"""

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox, IntSlider, Dropdown
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

# === الثوابت الفيزيائية ===
h, m, c, L_total = 6.626e-34, 9.109e-31, 3e8, 2.2

def de_broglie_wavelength(v):
    return h / (m * v)

def double_slit_intensity_by_wavelength(x, wavelength, L, a_width, d_slit):
    beta = (np.pi * d_slit * x) / (wavelength * L)
    alpha = (np.pi * a_width * x) / (wavelength * L)
    return np.cos(beta)**2 * np.sinc(alpha / np.pi)**2

def double_slit_intensity_with_spread(x, v_mean, delta_v, L, a_width, d_slit, n_samples=100):
    velocities = np.random.normal(v_mean, delta_v, n_samples)
    total = np.zeros_like(x)
    for v in velocities:
        lam = de_broglie_wavelength(v)
        total += double_slit_intensity_by_wavelength(x, lam, L, a_width, d_slit)
    return total / n_samples

def particle_like_pattern(x, wavelength, L, a_width, d_slit):
    sigma = a_width * L / wavelength
    return 0.5 * (np.exp(-(x + d_slit/2)**2 / (2 * sigma**2)) + np.exp(-(x - d_slit/2)**2 / (2 * sigma**2)))

def compute_visibility(x, I):
    peaks, _ = find_peaks(I, distance=len(x)//30)
    if len(peaks) < 2: return 0.0
    I_max, center_idx = np.max(I[peaks]), np.argmin(np.abs(x))
    search = np.where(np.abs(x - x[center_idx]) < 5e-3)[0]
    I_min = np.min(I[search]) if len(search) > 0 else np.min(I)
    return (I_max - I_min) / (I_max + I_min) if (I_max + I_min) > 0 else 0

def jonsson_validation(v_mean, a_width, d_slit, L):
    lam = de_broglie_wavelength(v_mean)
    theo_min = lam * L / a_width * 1000
    exp_min = 0.18
    error = abs(theo_min - exp_min) / exp_min * 100
    print(f"\n🔬 التحقق: الخطأ النسبي {error:.1f}% | النظري {theo_min:.3f}mm | التجريبي {exp_min}mm")
    return error

def white_light_pattern_rgb(x, L, a_width, d_slit):
    lams = [650e-9, 532e-9, 450e-9]
    ints = [double_slit_intensity_by_wavelength(x, l, L, a_width, d_slit) for l in lams]
    return [i/np.max(i) if np.max(i)>0 else i for i in ints]

@interact(
    mode=Dropdown(options=['Electron (de Broglie) - إلكترون', 'Photon (monochromatic) - فوتون أحادي اللون', 'White light (RGB) - ضوء أبيض ملون'], value='Electron (de Broglie) - إلكترون', description='نمط المحاكاة'),
    v_mean=FloatSlider(value=5.8e5, min=2e5, max=1.2e6, description='سرعة الإلكترون'),
    delta_v=FloatSlider(value=0.0, min=0.0, max=2e5, description='انتشار السرعة'),
    wavelength_nm=FloatSlider(value=532, min=380, max=750, description='الطول الموجي'),
    a_width=FloatSlider(value=0.72e-6, min=0.2e-6, max=1.5e-6, description='عرض الشق'),
    d_slit=FloatSlider(value=2.45e-6, min=0.5e-6, max=2e-3, description='المسافة d'),
    observer_active=Checkbox(value=False, description='مكتشف المسار'),
    meas_strength=FloatSlider(value=0.0, min=0.0, max=1.0, description='قوة القياس'),
    temperature=FloatSlider(value=0.0, min=0, max=1000, description='ضجيج (K)'),
    show_buildup=Checkbox(value=False, description='تراكم تدريجي'),
    n_particles=IntSlider(value=300, min=50, max=1000, description='عدد الجسيمات')
)
def interactive_lab(mode, v_mean, delta_v, wavelength_nm, a_width, d_slit, observer_active, meas_strength, temperature, show_buildup, n_particles):
    if 'Photon' in mode: lam = wavelength_nm * 1e-9
    elif 'Electron' in mode: lam = de_broglie_wavelength(v_mean)
    else: lam = 532e-9
    
    spacing = lam * L_total / d_slit
    x_limit = max(0.005, 3 * spacing)
    x = np.linspace(-x_limit, x_limit, 1500)

    if 'Electron' in mode:
        I_interf = double_slit_intensity_with_spread(x, v_mean, delta_v, L_total, a_width, d_slit) if delta_v > 0 else double_slit_intensity_by_wavelength(x, lam, L_total, a_width, d_slit)
        if abs(v_mean - 70e3) < 20e3: jonsson_validation(v_mean, a_width, d_slit, L_total)
    elif 'Photon' in mode: I_interf = double_slit_intensity_by_wavelength(x, lam, L_total, a_width, d_slit)
    else: 
        I_R, I_G, I_B = white_light_pattern_rgb(x, L_total, a_width, d_slit)
        I_interf = (I_R + I_G + I_B) / 3.0

    I_particle = particle_like_pattern(x, lam, L_total, a_width, d_slit)
    I = ((1 - meas_strength) * I_interf + meas_strength * I_particle) if observer_active else I_interf
    if temperature > 0:
        I += np.random.normal(0, (temperature/1000.0)*0.15*np.max(I), len(I))
        I = np.maximum(I, 0)
    I /= np.max(I)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(x*1000, I)
    ax1.set_title(f'Pattern Analysis | Visibility: {compute_visibility(x, I):.1%}')
    if 'White' in mode:
        I_Rw, I_Gw, I_Bw = white_light_pattern_rgb(x, L_total, a_width, d_slit)
        scr = np.zeros((100, len(x), 3))
        scr[:,:,0], scr[:,:,1], scr[:,:,2] = I_Rw, I_Gw, I_Bw
        ax2.imshow(scr, aspect='auto', extent=[-x_limit*1000, x_limit*1000, 0, 1])
    else:
        ax2.imshow(np.tile(I, (100, 1)), cmap='hot', aspect='auto', extent=[-x_limit*1000, x_limit*1000, 0, 1])
    plt.show()